# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. [1. 2. 3.] [0 1 1] | Деление массива при обычном / NumPy делает числа дробными с точкой на конце. // деление с отбрасыванием дробной части, остаётся только целая часть
2. [4. 4. 4.] |  Когда мы делаем .sum(axis=1) без параметра keepdims=True, NumPy схлопывает 4 столбца и превращает результат в обычный одномерный массив (строку) из трёх четвёрок (форма (3,)), а не вертикальный столбец (3, 1)
3. [[0,1,2], [1,2,3], [2,3,4]] | Встречный broadcasting — формы (3,) и (3, 1) совместимы, поэтому строка [0, 1, 2] повторяется вниз, а столбец [[0], [1], [2]] повторяется вправо до матрицы (3, 3), потом выполняется сложение 
4. 6.0 | извлекаем квадрат из каждого числа в массиве, затем суммируем все три числа
5. 7.5  | выводит массив [1, 2, 3, 4] возводим числа в массиве в квадрат и находим среднее арифметическое ,сложил все числа и поделил на количество 

In [3]:
import numpy as np
a = np.array([2, 4, 6]); print(a / 2, a // 4)


[1. 2. 3.] [0 1 1]


In [4]:
import numpy as np
m = np.ones((3, 4)); print(m.sum(axis=1))

[4. 4. 4.]


In [5]:
import numpy as np
print(np.arange(3) + np.arange(3).reshape(3, 1))

[[0 1 2]
 [1 2 3]
 [2 3 4]]


In [6]:
import numpy as np
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())

6.0


In [7]:
import numpy as np
print((np.arange(1, 5) ** 2).mean())

7.5


In [6]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [7]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [12]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [16]:
# 1
print(clean.sum())

4739


In [9]:
sep = amount[clean & (month == '2026-09')].sum()
octb = amount[clean & (month == '2026-10')].sum()
print(sep)
print(octb)
print(f"{(octb / sep - 1) * 100:.2f}%")

35243410.0
34419520.0
-2.34%


У меня не совпало, потому что я не почистил список при выполнении ДЗ2

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [34]:
# 2
import numpy as np
categ = np.unique(cat)
months = np.array(["2026-09", "2026-10"])

# Таблица 5 × 2: строка = категория, столбец = месяц
table = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months]
                  for c in categ])
for c, r in zip(categ, table):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")
print(table.shape)
total_cat = table.sum(axis=1)

print(categ[total_cat.argmax()])
print(np.round((table[:, 1] / table[:, 0] - 1) * 100))

Дом           4,997,730    5,125,790
Книги         1,065,300    1,140,910
Красота       2,610,950    2,678,990
Одежда        9,167,100    8,505,730
Электроника   17,402,330   16,968,100
(5, 2)
Электроника
[ 3.  7.  3. -7. -2.]


Больше всего выручки приносит за 2 месяца Электроника. Сильнее всех упала Одежда  -7%

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [46]:
# 3
import numpy as np
categ = np.unique(cat)
months = np.array(["2026-09", "2026-10"])

counts = np.array([[(clean & (cat == c) & (month == m)).sum() for m in months]
                  for c in categ])
for c, r in zip(categ, counts):
    print(f"{c:<12} {r[0]:>12,.0f} {r[1]:>12,.0f}")
print(counts.shape)
avg_check = table / counts
print(np.round(avg_check))
print(categ[avg_check[:,1].argmax()])

Дом                   476          476
Книги                 265          299
Красота               422          435
Одежда                678          652
Электроника           531          505
(5, 2)
[[10499. 10768.]
 [ 4020.  3816.]
 [ 6187.  6159.]
 [13521. 13046.]
 [32773. 33600.]]
Электроника


Средний чек самый высокий в октябре у Электроники

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [58]:
# 4
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])

# Таблица 6 × 2: строка = город, столбец = месяц
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])
for c, r in zip(cities, table):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")
print(table.shape)
share = table / table.sum(axis=0)
print(share)
print(share.sum(axis=0))

shym = cities == 'Шымкент'
sh = share[shym]
print(np.round((sh[0, 1] - sh[0, 0]) * 100))
print(np.round((sh[0, 1] / sh[0, 0] - 1) * 100))


Актобе        2,576,960    2,540,420
Алматы       14,501,070   14,643,020
Астана        9,034,220    8,850,530
Атырау          923,990      969,860
Караганда     3,648,520    3,371,680
Шымкент       4,558,650    4,044,010
(6, 2)
[[0.07311892 0.07380754]
 [0.41145479 0.42542778]
 [0.25633785 0.25713694]
 [0.02621738 0.02817762]
 [0.10352347 0.09795837]
 [0.12934759 0.11749176]]
[1. 1.]
-1.0
-9.0


Доля Шымкента в общей выручке снизилась с 12,9% до 11,7% (на 1,2 п.п., или на 9,2% относительно сентября)

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [66]:
# 5
diff_city = table[:,1] - table[:,0]
print(diff_city)
total_diff = diff_city.sum()
print(total_diff)
contrib = diff_city / total_diff * 100
print(contrib)
for c, val in zip(cities, contrib):
    print(f"{c:<10} {val:>12,.0f}")

[ -36540.  141950. -183690.   45870. -276840. -514640.]
-823890.0
[  4.43505808 -17.22924177  22.29545206  -5.56749081  33.60157303
  62.46464941]
Актобе                4
Алматы              -17
Астана               22
Атырау               -6
Караганда            34
Шымкент              62


Шымкент дает 62% падения выручки. В Шымкенте искать причину падения.

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [67]:
# 6
share = table / table.sum(axis=1, keepdims=True)
print(share)

[[0.50357019 0.49642981]
 [0.49756469 0.50243531]
 [0.50513538 0.49486462]
 [0.48788975 0.51211025]
 [0.51971739 0.48028261]
 [0.52991168 0.47008832]]


Без keepdims=True NumPy не может поделить матрицу (6,2) на (6,) и поэтому падает

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8